In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
import pandas as pd
customers=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv')
sellers=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv')
order_reviews=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv')
order_items=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv')
products=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv')
geolocations=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv')
category_name_translations=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv')
orders=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv')
order_payments=pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv')


In [3]:

order_reviews['review_creation_date']=pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp']=pd.to_datetime(order_reviews['review_answer_timestamp'])
order_items['shipping_limit_date']=pd.to_datetime(order_items['shipping_limit_date'])
orders['order_delivered_carrier_date']=pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_approved_at']=pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_customer_date']=pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_purchase_timestamp']=pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_estimated_delivery_date']=pd.to_datetime(orders['order_estimated_delivery_date'])



In [4]:
tables={'customers':customers,'sellers':sellers,'order_reviews':order_reviews,'order_items':order_items,'products':products,'geolocations':geolocations,'category_name_translations':category_name_translations,'orders':orders,'order_payments':order_payments}
for name,table in tables.items():
    print(name,table.shape)

customers (99441, 5)
sellers (3095, 4)
order_reviews (99224, 7)
order_items (112650, 7)
products (32951, 9)
geolocations (1000163, 5)
category_name_translations (71, 2)
orders (99441, 8)
order_payments (103886, 5)


In [5]:
tables={'customers':customers,'sellers':sellers,'order_reviews':order_reviews,'order_items':order_items,'products':products,'geolocations':geolocations,'category_name_translations':category_name_translations,'orders':orders,'order_payments':order_payments}
for name,table in tables.items():
    for column in table:
       print(name,column,table[column].isna().sum(),(table[column].isna().sum()/len(table)*100))

customers customer_id 0 0.0
customers customer_unique_id 0 0.0
customers customer_zip_code_prefix 0 0.0
customers customer_city 0 0.0
customers customer_state 0 0.0
sellers seller_id 0 0.0
sellers seller_zip_code_prefix 0 0.0
sellers seller_city 0 0.0
sellers seller_state 0 0.0
order_reviews review_id 0 0.0
order_reviews order_id 0 0.0
order_reviews review_score 0 0.0
order_reviews review_comment_title 87656 88.34153027493349
order_reviews review_comment_message 58247 58.70253164556962
order_reviews review_creation_date 0 0.0
order_reviews review_answer_timestamp 0 0.0
order_items order_id 0 0.0
order_items order_item_id 0 0.0
order_items product_id 0 0.0
order_items seller_id 0 0.0
order_items shipping_limit_date 0 0.0
order_items price 0 0.0
order_items freight_value 0 0.0
products product_id 0 0.0
products product_category_name 610 1.8512336499650999
products product_name_lenght 610 1.8512336499650999
products product_description_lenght 610 1.8512336499650999
products product_photos

In [6]:
import duckdb as db
test1=db.query("""
select order_status,count(order_status) as order_status_count
from orders
where order_delivered_customer_date is null
group by order_status


""").df()
print(test1)

  order_status  order_status_count
0      shipped                1107
1   processing                 301
2      created                   5
3    delivered                   8
4  unavailable                 609
5     canceled                 619
6     invoiced                 314
7     approved                   2


In [7]:
import duckdb as db
test2=db.query("""
select order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
from orders
where order_status='delivered' and order_delivered_customer_date is null

""").df()
print(test2)

                           order_id order_status order_purchase_timestamp  \
0  2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07   
1  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43   
2  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11   
3  e69f75a717d64fc5ecdfae42b2e8e086    delivered      2018-07-01 22:05:55   
4  0d3268bad9b086af767785e3f0fc0133    delivered      2018-07-01 21:14:02   
5  2d858f451373b04fb5c984a1cc2defaf    delivered      2017-05-25 23:22:43   
6  ab7c89dc1bf4a1ead9d6ec1ec8968a84    delivered      2018-06-08 12:09:39   
7  20edc82cf5400ce95e1afacc25798b31    delivered      2018-06-27 16:09:12   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-11-30 18:12:23                           NaT   
1          2018-06-25 08:05:00                           NaT   
2          2018-07-03 13:57:00                           NaT   
3          2018-07-03 13:57:00                    

In [8]:
import duckdb as db
test3=db.query("""
select order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,case when order_status='canceled' and order_delivered_customer_date is not null then 'flag' end as descision
from orders
where order_status='canceled' and order_delivered_customer_date is  not null

""").df()
test3.head()

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,descision
0,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-03-21 22:03:51,flag
1,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-16 14:36:59,flag
2,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-14 15:07:11,flag
3,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-19 18:47:43,flag
4,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-11-08 10:58:34,flag


In [9]:
import duckdb as db
test4=db.query("""
select o.order_id,o.order_status,o.order_delivered_customer_date,ore.review_comment_message,ore.review_id,ore.review_score,ore.review_creation_date
from orders o 
left join order_reviews ore on o.order_id=ore.order_id
where o.order_status='delivered' and o.order_delivered_customer_date is null

""").df()
test4.head(10)

,order_id,order_status,order_delivered_customer_date,review_comment_message,review_id,review_score,review_creation_date
0,0d3268bad9b086af767785e3f0fc0133,delivered,NaT,O produto chegou muito antes do prazo previsto...,ee2d30652e2f7fc00861074f795f5bf0,5,2018-07-06
1,e69f75a717d64fc5ecdfae42b2e8e086,delivered,NaT,None,bb311d9562ecbefc8e4be756d8999892,5,2018-07-07
2,f5dd62b788049ad9fc0526e3ad11a097,delivered,NaT,"Produto novo, muito bom.",c0dd6bec0375c376f044af102118526f,5,2018-06-29
3,2d858f451373b04fb5c984a1cc2defaf,delivered,NaT,None,4e755f114e50d33b9ac6a56e0d7d3ea9,5,2017-06-25
4,2ebdfc4f15f23b91474edf87475f108e,delivered,NaT,None,25e11638a3d01a87e8e62338a39eee28,5,2018-07-11
5,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,NaT,Chegou rápido tudo ok,f48c6c944a5d52dcca8ac5c4ec417cf2,5,2017-12-19
6,20edc82cf5400ce95e1afacc25798b31,delivered,NaT,Adorei,d055795a562efffefe47ef81e5435322,5,2018-07-06
7,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,NaT,Comprei um produto de uma marca e recebi outro...,0d4c56af896dd6eb9de8edbaa1902d22,1,2018-06-16


In [10]:
import pandas as pd
products=products.rename(columns={'product_name_lenght':'product_name_length','product_description_lenght':'product_description_length'})
products.head(5)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [11]:
import duckdb as db
test5=db.query("""
select product_category_name,product_name_length,product_description_length,product_photos_qty,count(*) as null_count
from products
where product_category_name is null and product_name_length is null and product_description_length is null and product_photos_qty is null
group by product_category_name,product_name_length,product_description_length,product_photos_qty

""").df()
test5.head(20)

,product_category_name,product_name_length,product_description_length,product_photos_qty,null_count
0,None,NaN,NaN,NaN,610


In [12]:
test5=db.query("""
select o.product_id,sum(o.price) as sum_price,p.product_category_name,p.product_name_length,p.product_description_length,p.product_photos_qty,count(*) as count
from order_items o 
left join products p on p.product_id=o.product_id
where p.product_category_name is null and p.product_name_length is null and p.product_description_length is  null and p.product_photos_qty is  null
group by o.product_id,p.product_category_name,p.product_name_length,p.product_description_length,p.product_photos_qty

""").df()
print(test5.describe)

<bound method NDFrame.describe of                            product_id  sum_price product_category_name  \
0    78e4af9e5ea7b8768765492fdfaae545      20.94                  None   
1    76d1a1a9d21ab677a61c3ae34b1b352f    5712.64                  None   
2    b1d207586fca400a2370d50a9ba1da98    7152.00                  None   
3    c30302f36e15f43c1b85c9cb002028d5      60.00                  None   
4    eaecac42c42a65691bc3662b123e884d      15.00                  None   
..                                ...        ...                   ...   
605  d2eb4e94aeb255a953cbd8b97361a848      17.99                  None   
606  3bc5164bc7f4be77002d6651da65c98c      22.50                  None   
607  817a68e75e8fc94fa28915d8169d67bf      89.90                  None   
608  8b67de5ab270587094f963bbed8b3d29    1599.00                  None   
609  48212f0e644cd07610881a3b6e497ac4      34.30                  None   

     product_name_length  product_description_length  product_photos_qty  \
0

In [13]:
test6=db.query("""
select o.product_id,sum(o.price) as sum ,p.product_category_name
from order_items o 
left join products p on p.product_id=o.product_id
where p.product_category_name is null
group by o.product_id,p.product_category_name
""").df()
test6.head()

,product_id,sum,product_category_name
0,5a848e4ab52fd5445cdc07aab1c40e48,24229.03,None
1,f55cd6a68f96c3041176998ce1850ff3,126.00,None
2,5eb564652db742ff8f28759cd8d2652a,563.00,None
3,3128a1273a207ae946c4dd1e23c86fed,305.00,None
4,80872f7f128200f12c5af9494b1d37a0,843.00,None


In [14]:
test7=db.query("""
select count(distinct p.product_id) AS products,count(*) as order_lines,sum(o.price) as total_revenue
from order_items o
left join products p on p.product_id = o.product_id
where p.product_category_name is null
""").df()
print(test7)

   products  order_lines  total_revenue
0       610         1603      179535.28


In [15]:
test8=db.query("""
select sum(case when p.product_category_name is null then o.price else 0 end)/sum(o.price) as share_percentage
from products p 
inner join order_items o on o.product_id=p.product_id

""").df()
print(test8)

   share_percentage
0          0.013209


In [16]:
test9=db.query("""select sum(case when review_comment_title is null then 1 else 0 end) as null_title_count,sum(case when review_comment_message is null then 1 else 0 end) as null_message_count,
sum(case when review_comment_title is not null then 1 else 0 end) as not_null_title_courevient,
sum(case when review_comment_message is not null then 1 else 0 end) as not_null_message_count
from order_reviews
""").df()
print(test9)

   null_title_count  null_message_count  not_null_title_courevient  \
0           87656.0             58247.0                    11568.0   

   not_null_message_count  
0                 40977.0  


In [17]:
test10=db.query("""
select sum(case when review_comment_title is not null and review_comment_message is not null then 1 else 0 end) as title_message_present,
sum(case when review_comment_title is not null and review_comment_message is  null then 1 else 0 end) as title_present_message_absent,
sum(case when review_comment_title is null and review_comment_message is not null then 1 else 0 end) as title_absent_message_present,
sum(case when review_comment_title is null and review_comment_message is null then 1 else 0 end) as title_message_absent
from order_reviews

""").df()
print(test10)

   title_message_present  title_present_message_absent  \
0                 9839.0                        1729.0   

   title_absent_message_present  title_message_absent  
0                       31138.0               56518.0  


In [18]:
test11=db.query('''
select product_id
from products
where product_weight_g is null and product_id in (select product_id from products where 
product_category_name is null and
product_name_length is null and
product_description_length is null and
product_photos_qty is null)

''').df()
print(test11)

                         product_id
0  5eb564652db742ff8f28759cd8d2652a


In [19]:
test12=db.query("""
select count(customer_id) as total_count,count(distinct customer_id) as distinct_count
from orders
""").df()
print(test12)

   total_count  distinct_count
0        99441           99441


In [20]:
test13=db.query("""
with new as(
select g.geolocation_zip_code_prefix,s.seller_zip_code_prefix
from geolocations g
left join sellers s on g.geolocation_zip_code_prefix=s.seller_zip_code_prefix
),
new2 as(
select count(geolocation_zip_code_prefix) as count1,count(seller_zip_code_prefix) as count2
from new
)
select count2/count1*100 as pct
from new2
""").df()
print(test13)

        pct
0  37.30061


In [21]:
test14=db.query("""
with new as(
select count(distinct geolocation_zip_code_prefix) as geo_count
from geolocations
),
new1 as(
select count(distinct seller_zip_code_prefix) as seller_zip_count
from sellers
)
select (seller_zip_count/geo_count)*100 as pct
from new,new1
""").df()
print(test14)

         pct
0  11.811728


In [22]:
test15=db.query("""
with new as(
select customer_state,count(customer_state) as customer_state_count
from customers
group by customer_state
),
new1 as(
select count(*) as total_count
from customers
)
select customer_state,(customer_state_count/total_count)*100 as pct
from new,new1
order by pct desc
""").df()
test15.head()

,customer_state,pct
0,SP,41.980672
1,RJ,12.924247
2,MG,11.700405
3,RS,5.496727
4,PR,5.073360


In [23]:
test16=db.query("""
with new as(
select customer_state,customer_city,count(customer_city) as city_count
from customers
group by customer_state,customer_city
),
new1 as(
select count(customer_city) as total_count
from customers
)
select customer_state,customer_city,(city_count/total_count)*100 as pct
from new,new1
order by pct desc

""").df()
test16.head(10)

,customer_state,customer_city,pct
0,SP,sao paulo,15.627357
1,RJ,rio de janeiro,6.920687
2,MG,belo horizonte,2.788588
3,DF,brasilia,2.142979
4,PR,curitiba,1.529550
5,SP,campinas,1.452117
6,RS,porto alegre,1.386752
7,BA,salvador,1.251999
8,SP,guarulhos,1.195684
9,SP,sao bernardo do campo,0.943273


In [24]:
test17=db.query("""
with new as(
select customer_zip_code_prefix	,count(distinct customer_zip_code_prefix) as count1
from customers
group by customer_zip_code_prefix
),
new1 as(
select count(distinct customer_zip_code_prefix) as total
from customers
)
select customer_zip_code_prefix	,(count1/total)*100 as pct
from new,new1
order by pct desc
""").df()
test17.head(10)

,customer_zip_code_prefix,pct
0,9121,0.006669
1,4571,0.006669
2,2075,0.006669
3,9890,0.006669
4,3636,0.006669
5,4710,0.006669
6,38410,0.006669
7,5874,0.006669
8,3563,0.006669
9,20030,0.006669


In [25]:
test18=db.query("""
with new as(
select count(distinct customer_zip_code_prefix) as count
from customers
),
new1 as(
select count(customer_id) as total_count
from customers
)
select (count/total_count)*100 as pct
from new,new1

""").df()
print(test18)

         pct
0  15.078288


In [26]:
products.head(1)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0


In [27]:
test19=db.query("""
select distinct p.product_id,p.product_category_name,p.product_name_length,p.product_description_length,p.product_photos_qty,p.product_weight_g,p.product_length_cm,
p.product_height_cm,p.product_width_cm,o.price
from products p
left join order_items o on o.product_id=p.product_id
where p.product_weight_g is null 

""").df()
test19.head(10)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,price
0,5eb564652db742ff8f28759cd8d2652a,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.0
1,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN,1934.0
2,5eb564652db742ff8f28759cd8d2652a,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0


In [28]:
test20=db.query("""
with new as(
select count(distinct customer_unique_id ) as count
from customers
),
new1 as(
select count(customer_id) as total_count
from customers
)
select (count/total_count)*100 as pct 
from new,new1
""").df()
test20.head()

,pct
0,96.636196


In [29]:
test20a=db.query("""
select count(customer_zip_code_prefix) as count,customer_zip_code_prefix,customer_city,customer_state
from customers
group by customer_zip_code_prefix,customer_city,customer_state
order by count desc
""").df()
test20a.head(20)

,count,customer_zip_code_prefix,customer_city,customer_state
0,142,22790,rio de janeiro,RJ
1,124,24220,niteroi,RJ
2,121,22793,rio de janeiro,RJ
3,117,24230,niteroi,RJ
4,110,22775,rio de janeiro,RJ
5,101,29101,vila velha,ES
6,95,13212,jundiai,SP
7,93,35162,ipatinga,MG
8,89,22631,rio de janeiro,RJ
9,87,38400,uberlandia,MG


In [30]:
test21=db.query("""
with new as(
select count( customer_zip_code_prefix) as count,customer_zip_code_prefix,customer_city,customer_state
from customers
where customer_city='rio de janeiro' or customer_city='sao paulo'
group by customer_zip_code_prefix,customer_city,customer_state
order by count desc
),
new1 as(
select count,customer_zip_code_prefix,customer_city,customer_state,dense_rank() over(partition by customer_city,customer_state order by count desc) as rnk
from new
)
select count,customer_zip_code_prefix,customer_city,customer_state
from new1
where rnk=1
""").df()
test21.head(20)

,count,customer_zip_code_prefix,customer_city,customer_state
0,36,8290,sao paulo,SP
1,142,22790,rio de janeiro,RJ


In [31]:
test22=db.query("""
select count(distinct customer_zip_code_prefix) as distinct_count,customer_city
from customers
where customer_city='rio de janeiro' or customer_city='sao paulo'
group by customer_city


""").df()
test22.head()

,distinct_count,customer_city
0,2843,sao paulo
1,373,rio de janeiro


In [32]:
test23=db.query("""
with new as(
select customer_zip_code_prefix,count(distinct customer_city) as count
from customers
group by customer_zip_code_prefix
having count>1
),
new1 as(
select  n.customer_zip_code_prefix,c.customer_city
from new n
inner join customers c on c.customer_zip_code_prefix=n.customer_zip_code_prefix
)
select * from new1
""").df()
test23.shape

(476, 2)

In [33]:
test24=db.query("""
select customer_zip_code_prefix,count(distinct customer_city) as count
from customers
group by customer_zip_code_prefix
having count>1

""").df()
test24.shape

(39, 2)

In [34]:
test25=db.query("""
with new as(
select customer_zip_code_prefix,
count(distinct customer_city) as count
from customers
group by customer_zip_code_prefix
having count>1
),
new1 as(
select c.customer_city,
n.customer_zip_code_prefix
from customers c 
inner join new n on c.customer_zip_code_prefix=n.customer_zip_code_prefix
)
select distinct customer_city,
customer_zip_code_prefix
from new1
order by customer_city asc
""").df()
test25.head(5)

,customer_city,customer_zip_code_prefix
0,abrantes,42840
1,apora,48355
2,arraial d ajuda,45816
3,arraial d'ajuda,45816
4,balneario picarras,88380


In [35]:
test26=db.query("""
with new as(
select seller_state,count(seller_state) as count
from sellers
group by seller_state
),
new1 as(
select count(*) as total_count
from sellers
)
select seller_state,(count/total_count)*100 as pct
from new,new1
order by pct desc
""").df()
test26.head(10)

,seller_state,pct
0,SP,59.741519
1,PR,11.276252
2,MG,7.883683
3,SC,6.138934
4,RJ,5.525040
5,RS,4.168013
6,GO,1.292407
7,DF,0.969305
8,ES,0.743134
9,BA,0.613893


In [36]:
test27=db.query("""
with new as(
select seller_city,count(seller_city) as count
from sellers
group by seller_city
),
new1 as(
select count(*) as total_count
from sellers
)
select seller_city,(count/total_count)*100 as pct
from new,new1
order by pct desc
""").df()
test27.head(10)

,seller_city,pct
0,sao paulo,22.423263
1,curitiba,4.103393
2,rio de janeiro,3.101777
3,belo horizonte,2.197092
4,ribeirao preto,1.680129
5,guarulhos,1.615509
6,ibitinga,1.583199
7,santo andre,1.453958
8,campinas,1.324717
9,maringa,1.292407


In [37]:
test28=db.query("""
select count(distinct seller_id) as distinct_seller_count
from sellers

""").df()
test28.head()

,distinct_seller_count
0,3095


In [38]:
test29=db.query("""
select customer_id,datediff('day',order_delivered_customer_date,order_delivered_carrier_date) as diff
from orders
where order_delivered_customer_date<order_delivered_carrier_date
order by diff desc
""").df()
test29.head(1)

,customer_id,diff
0,e37d47e7eec62f08dc5deecc7d5532d6,16


In [39]:
test30=db.query("""
with new as(
select distinct ore.order_id,ore.review_score,oi.product_id
from order_reviews ore
inner join order_items oi on oi.order_id=ore.order_id
),
new1 as(
select n.order_id,n.review_score,p.product_id,p.product_category_name
from new n 
inner join products p on p.product_id=n.product_id
),
new2 as(
select product_category_name,count(product_category_name) as count
from new1
where review_score<=2
group by product_category_name
),
new3 as(
select product_category_name,count(product_category_name) as total_count
from new1
group by product_category_name
)
select n2.product_category_name,(n2.count/nullif(n3.total_count,0))*100 as pct
from new2 n2 
inner join new3 n3 on n2.product_category_name=n3.product_category_name

order by pct desc

""").df()
test30.head(20)

,product_category_name,pct
0,seguros_e_servicos,50.000000
1,pc_gamer,37.500000
2,portateis_cozinha_e_preparadores_de_alimentos,28.571429
3,fashion_roupa_masculina,27.868852
4,moveis_escritorio,23.456790
5,audio,22.126437
6,fashion_roupa_feminina,21.951220
7,casa_conforto_2,21.739130
8,moveis_colchao_e_estofado,21.052632
9,construcao_ferramentas_seguranca,20.238095


In [40]:
ntest1=db.query("""
select count(distinct product_category_name) as count
from products

""").df()
ntest1.head()

,count
0,73


In [41]:
test31=db.query("""
with new as(
select ore.order_id,ore.review_score,oi.product_id
from order_reviews ore
inner join order_items oi on ore.order_id=oi.order_id
where ore.review_score<=2
),
new1 as(
select  n.order_id,n.review_score,n.product_id,p.product_category_name
from new n 
left join products p on p.product_id=n.product_id
),
new2 as(
select n1.order_id,n1.review_score,n1.product_id,c.product_category_name_english
from new1 n1 
left join category_name_translations c on c.product_category_name=n1.product_category_name
),
new3 as(
select product_category_name_english,count(product_category_name_english) as count
from new2
group by product_category_name_english
),
new4 as(
select count(*) as total_count
from new2
)
select product_category_name_english,(count/total_count)*100 as pct,count
from new3,new4
order by pct desc

""").df()
test31.head(10)

,product_category_name_english,pct,count
0,bed_bath_table,11.662709,2112
1,furniture_decor,8.951350,1621
2,computers_accessories,8.067812,1461
3,health_beauty,7.294715,1321
4,sports_leisure,6.957866,1260
5,housewares,5.997018,1086
6,watches_gifts,5.339886,967
7,telephony,4.202330,761
8,garden_tools,3.843393,696
9,auto,3.550721,643


In [42]:
test32=db.query("""
with new as(
select distinct ore.order_id,ore.review_score,oi.product_id
from order_reviews ore
inner join order_items oi on ore.order_id=oi.order_id
),
new1 as(
select n.order_id,n.review_score,n.product_id,p.product_category_name
from new n 
left join products p on p.product_id=n.product_id
),
new2 as(
select n1.order_id,n1.review_score,n1.product_id,c.product_category_name_english
from new1 n1 
left join category_name_translations c on c.product_category_name=n1.product_category_name
),
new3 as(
select product_category_name_english,sum(case when review_score<=2 then 1 else 0 end) as less_than_2 ,count(*) as total_count
from new2
group by product_category_name_english
)
select product_category_name_english,(less_than_2/total_count)* 100 as pct,total_count
from new3
order by pct desc
""").df()
test32.head(10)

,product_category_name_english,pct,total_count
0,security_and_services,50.000000,2
1,fashion_male_clothing,27.868852,122
2,office_furniture,23.456790,1296
3,audio,22.126437,348
4,fashio_female_clothing,21.951220,41
5,home_comfort_2,21.739130,23
6,furniture_mattress_and_upholstery,21.052632,38
7,None,20.488467,1474
8,construction_tools_safety,20.238095,168
9,home_confort,19.704433,406


In [43]:
test33=db.query("""
with new as(
select review_id,review_comment_message
from order_reviews
where review_comment_message=' ' or review_comment_message='  '
),
new1 as(
select review_id,case when review_comment_message=' ' or review_comment_message='  ' then null else review_comment_message end as review_comment_message
from new
)
select *  from new1
""").df()
test33.head(20)

,review_id,review_comment_message
0,54e5ac041fa070ccb4db7595b2e451ca,None
1,380e63e57f44120efd2184be29cd8a98,None
2,63c112c4b3772ebf732bc62d98b971b5,None
3,04bb4510dfa4af74e357521612d82ae9,None
4,8e2d665afbaf7f089558d04f6d4ce7f9,None
5,f476134a5846b7fc4a1fdf587353a2c0,None
6,db03b06904bc8618927edd34cab89e4c,None
7,02a630a3e22f98f1e054190a84a067b5,None
8,6831c52a8625e94f8b8fc553dbac12df,None


In [44]:
test34=db.query("""
with new as(
select  distinct c.customer_id,c.customer_city,c.customer_state,o.order_id
from customers c 
inner join orders o on c.customer_id=o.customer_id
),
new1 as(
select  n.customer_id,n.customer_city,n.customer_state,n.order_id,oi.product_id,oi.seller_id
from new n 
inner join order_items oi on oi.order_id=n.order_id
),
new2 as(
select n1.customer_id,n1.customer_city,n1.customer_state,n1.order_id,p.product_category_name,n1.seller_id
from new1 n1 
left join products p on p.product_id=n1.product_id
),
new3 as(
select distinct n2.customer_id,n2.customer_city,n2.customer_state,n2.order_id,c.product_category_name_english,n2.seller_id
from new2 n2
left join category_name_translations c on n2.product_category_name=c.product_category_name
),
new4 as(
select n3.customer_Id,n3.customer_city,n3.customer_state,n3.order_id,n3.product_category_name_english,n3.seller_id
from new3 n3
),
new5 as(
select  n4.customer_city,n4.customer_state,n4.product_category_name_english,s.seller_city,s.seller_state,n4.order_id
from new4 n4
left join sellers s on s.seller_id=n4.seller_id
),
new6 as(
select n5.customer_city,n5.customer_state,n5.product_category_name_english,ore.review_score,n5.seller_city,n5.seller_state
from new5 n5 
left join order_reviews ore on ore.order_id=n5.order_id
),
new7 as(
select customer_city,customer_state,product_category_name_english,sum(case when review_score<=2 then 1 else 0 end) as less_than_2,count(*) as total_count,
seller_city,seller_state
from new6
group by customer_city,customer_state,product_category_name_english,seller_city,seller_state
)
select customer_city,customer_state,product_category_name_english,(less_than_2*100.0/total_count) as pct,total_count,seller_state,seller_city
from new7
order by total_count desc
""").df()
test34.head(10)

,customer_city,customer_state,product_category_name_english,pct,total_count,seller_state,seller_city
0,sao paulo,SP,bed_bath_table,18.131868,910,SP,ibitinga
1,rio de janeiro,RJ,bed_bath_table,28.501229,407,SP,ibitinga
2,sao paulo,SP,telephony,12.868633,373,SP,sao paulo
3,sao paulo,SP,health_beauty,9.742120,349,SP,sao paulo
4,sao paulo,SP,housewares,13.888889,324,SP,sao paulo
5,sao paulo,SP,sports_leisure,7.301587,315,SP,sao paulo
6,sao paulo,SP,furniture_decor,12.794613,297,SP,sao paulo
7,sao paulo,SP,watches_gifts,15.079365,252,SP,sao paulo
8,sao paulo,SP,toys,12.727273,220,SP,sao paulo
9,sao paulo,SP,bed_bath_table,8.715596,218,SP,piracicaba


In [45]:
test35=db.query("""
with new as(
select o.order_id,ore.review_score,o.order_delivered_customer_date,order_estimated_delivery_date
from orders o 
inner join order_reviews ore on ore.order_id=o.order_id
),
new1 as(
select order_id,review_score,datediff('day',order_estimated_delivery_date,order_delivered_customer_date) as diff
from new
),
new2 as(
select order_id,(review_score),(case when diff>0 then 1 else null end) as diff_gre_count,
from new1
),new3 as(
select review_score,count(diff_gre_count) as count,sum(case when review_score=5 then 1 else 0 end) as review_5
from new2
where review_score=5
group by review_score
)
select review_score,count/review_5*100
from new3
""").df()
test35.head()

,review_score,((count / review_5) * 100)
0,5,1.849009


In [46]:
test36=db.query("""
with new as(
select o.order_id,ore.review_score,(case when (cast(o.order_delivered_customer_date as date)>cast(o.order_estimated_delivery_date as date)) then 'late'
when (cast(o.order_delivered_customer_date as date)<=cast(o.order_estimated_delivery_date as date)) then 'not_late'when o.order_delivered_customer_date is null then 'date_not_available' end ) as status_checker
from orders o 
inner join order_reviews ore on o.order_id=ore.order_id
),
new1 as(
select status_checker,count(status_checker) as count,sum(case when review_score<=2 then 1 else 0 end) as less_than_2
from new
group by status_checker
)
select status_checker,(less_than_2*100.0/count) as pct,count
from new1
""").df()
test36.head()

,status_checker,pct,count
0,not_late,9.283038,89949
1,date_not_available,77.661431,2865
2,late,62.402496,6410


In [47]:
test37=db.query("""
with new as(
select order_delivered_customer_date,order_estimated_delivery_date,(cast(order_delivered_customer_date as date)-cast(order_estimated_delivery_date as date)) as diff
from orders
where order_delivered_customer_date is not null
),
new1 as(
select case when diff<=0 then 'on_time' when diff>=1 and diff<=3 then 'slight_late' when diff>=4 and diff<=7 then 'delay' when diff>7 then 'huge_delay' end as status
from new
),
new2 as(
select count(*) as total_count
from new1
)
select status,count(status)*100.0/total_count as pct,count(status) as count,total_count
from new1,new2
group by status,total_count

""").df()
test37.head()

,status,pct,count,total_count
0,huge_delay,2.967577,2863,96476
1,on_time,93.226295,89941,96476
2,slight_late,1.938306,1870,96476
3,delay,1.867822,1802,96476


In [48]:
test38=db.query("""
with new as(
select o.order_delivered_customer_date,o.order_estimated_delivery_date,(cast(o.order_delivered_customer_date as date)-cast(o.order_estimated_delivery_date as date)) as diff,
ore.review_score
from orders o
inner join order_reviews ore on o.order_id=ore.order_id
where order_delivered_customer_date is not null 
),
new1 as(
select case when diff<=0 then 'on_time' when diff>=1 and diff<=3 then 'slight_late' when diff>=4 and diff<=7 then 'delay' when diff>7 then 'huge_delay' end as status,review_score
from new
),
new2 as(
select status,sum(case when review_score<=2 then 1 else 0 end) as less_than_2 ,count(*) as total_count
from new1
group by status
)
select status,(less_than_2*100.0/total_count) as pct,less_than_2,total_count
from new2
""").df()
test38.head()

,status,pct,less_than_2,total_count
0,on_time,9.283038,8350.0,89949
1,slight_late,32.165948,597.0,1856
2,huge_delay,79.163688,2215.0,2798
3,delay,67.653759,1188.0,1756


In [49]:
full_carrier_timestamp=orders['order_delivered_carrier_date']
full_delivered_timestamp=orders['order_delivered_customer_date']

In [50]:
exp=db.query("""
select order_delivered_carrier_date,order_delivered_customer_date
from orders
 where order_delivered_carrier_date > order_delivered_customer_date or (order_delivered_carrier_date is null and order_delivered_customer_date is not null)
""").df()
print(exp.shape)

(24, 2)


In [51]:
import numpy as np
import pandas as pd
delivered_date = pd.to_datetime(orders['order_delivered_customer_date'])
carrier_date= pd.to_datetime(orders['order_delivered_carrier_date'])
estimated_date=pd.to_datetime(orders['order_estimated_delivery_date'])


In [52]:
# conditions=[((orders['order_status']=='delivered')&(orders['order_delivered_customer_date'].isnull())),
#              ((orders['order_status']=='canceled')&orders['order_delivered_customer_date'].notnull()),
#               (((carrier_date.notnull()&delivered_date.notnull())) & (carrier_date>delivered_date)),
#                ((delivered_date.notnull()&estimated_date.notnull())&(delivered_date>estimated_date)),
#            ((delivered_date.notnull()&estimated_date.notnull())&(delivered_date<=estimated_date))]
# choices=['delivered but no date','canceled but has date','carrier date before delivered','late','not late']
# orders['flag']=np.select(conditions,choices,default='ok')

In [53]:
conditions=[delivered_date.isna(),(delivered_date.dt.date>estimated_date.dt.date),(delivered_date.dt.date<=estimated_date.dt.date)]
choices=['data_not_available','late','not late']
orders['delivery_status']=np.select(conditions,choices,default='unknown')

In [54]:
exp1=db.query("""
select delivery_status,count(delivery_status) as count
from orders
group by delivery_status
""").df()
exp1.head()

,delivery_status,count
0,not late,89941
1,data_not_available,2965
2,late,6535


In [55]:
orders['canceled_but_has_date']=np.where(((orders['order_status']=='canceled') & (delivered_date.notnull())),'yes','no')

In [56]:
exp2=db.query("""
select canceled_but_has_date,count(canceled_but_has_date) as count
from orders
where canceled_but_has_date='yes'
group by canceled_but_has_date
""").df()
exp2.head()

,canceled_but_has_date,count
0,yes,6


In [57]:
orders['delivered_but_has_nodate']=np.where((orders['order_status']=='delivered')&(delivered_date.isnull()),'yes','no')

In [58]:
exp3=db.query("""
select delivered_but_has_nodate,count(delivered_but_has_nodate) as count
from orders
where delivered_but_has_nodate='yes'
group by delivered_but_has_nodate
""").df()
exp3.head()

,delivered_but_has_nodate,count
0,yes,8


In [59]:
orders['carrier_date_before_delivered_date']=np.where((carrier_date>delivered_date),'yes','no')

In [60]:
exp4=db.query("""
select carrier_date_before_delivered_date,count(carrier_date_before_delivered_date) as count
from orders
where carrier_date_before_delivered_date='yes'
group by carrier_date_before_delivered_date
""").df()
exp4.head()

,carrier_date_before_delivered_date,count
0,yes,23


In [61]:
exp5=db.query("""
select customer_id
from orders
where carrier_date_before_delivered_date='yes'
""").df()
print(exp5.shape)


(23, 1)


In [62]:
test39=db.query("""
with new as(
select  oi.order_id,oi.seller_id,oi.price,oi.freight_value,c.customer_city,c.customer_state,s.seller_city,s.seller_state
from order_items oi 
inner join orders o on o.order_id=oi.order_id
inner join customers c on c.customer_id=o.customer_id
inner join sellers s on s.seller_id=oi.seller_id
where o.order_status not in ('canceled','unavailable')
)
select customer_state,customer_city,seller_state,seller_city,sum(freight_value) as freight_sum,(sum(price)+sum(freight_value)) as total_sum,(sum(freight_value)*100.0/(sum(price)+sum(freight_value))) as pct,count(*) as total_count,
max(price) as max_price,round(avg(price),2) as avg_price
from new
group by customer_city,seller_city,seller_state,customer_state
having total_count>=20
order by pct desc
""").df()
test39.head(20)

,customer_state,customer_city,seller_state,seller_city,freight_sum,total_sum,pct,total_count,max_price,avg_price
0,SP,sao paulo,SP,dracena,577.40,1742.10,33.143907,48,34.20,24.26
1,PE,caruaru,SP,sao paulo,1185.93,3731.50,31.781589,32,225.00,79.55
2,PE,recife,SP,ibitinga,1007.87,3174.75,31.746437,28,230.00,77.39
3,SP,sao paulo,RJ,tres rios,274.63,917.98,29.916774,20,99.99,32.17
4,RJ,rio de janeiro,SP,poa,348.87,1202.44,29.013506,20,54.90,42.68
5,RJ,rio de janeiro,SC,canoinhas,404.54,1403.84,28.816674,22,96.00,45.42
6,SP,sao paulo,SC,canoinhas,354.14,1233.94,28.699937,20,96.00,43.99
7,BA,salvador,MG,ilicinea,1259.23,4412.05,28.540701,27,179.99,116.77
8,RJ,rio de janeiro,SP,atibaia,404.30,1423.30,28.405817,20,129.99,50.95
9,SP,sao paulo,SP,porto ferreira,718.27,2552.91,28.135344,35,240.00,52.42


In [63]:
products['missed_category_name']=np.where((products['product_category_name'].isnull())
                                          &(products['product_name_length'].isnull())
                                           &(products['product_description_length'].isnull())
                                           &(products['product_photos_qty'].isnull()),'yes','no')

In [64]:
exp6=db.query("""
select product_id
from products
where missed_category_name='yes'
""").df()
print(exp6.shape)

(610, 1)


In [65]:
products['missing_dimensions']=np.where((products['product_weight_g'].isnull())
                                             &(products['product_length_cm'].isnull())
                                             &(products['product_height_cm'].isnull())
                                             &(products['product_width_cm'].isnull()),'yes','no' )

In [66]:
exp9=db.query("""
select product_id
from products
where missing_dimensions='yes'
""").df()
exp9.head()

,product_id
0,09ff539a621711667c43eba6a3bd8466
1,5eb564652db742ff8f28759cd8d2652a


In [67]:
exp10=db.query("""
select product_id
from products
where missed_category_name='yes' and missing_dimensions='yes'
""").df()
exp10.head()

,product_id
0,5eb564652db742ff8f28759cd8d2652a


In [68]:
products['product_category_name']=products['product_category_name'].fillna('unknown')

In [69]:
exp11=db.query("""
select product_category_name
from products
where product_category_name='unknown'
""").df()
print(exp11.shape)

(610, 1)


In [70]:
exp11=db.query("""
select product_id
from products
where missing_dimensions='yes'
""").df()
exp11.head()

,product_id
0,09ff539a621711667c43eba6a3bd8466
1,5eb564652db742ff8f28759cd8d2652a


In [71]:
order_reviews['review_comment_title']=order_reviews['review_comment_title'].fillna('no_title')
order_reviews['review_comment_title']=order_reviews['review_comment_title'].replace(['',' ','  '],'no_title')

In [72]:
exp12=db.query("""
select review_comment_title
from order_reviews
where review_comment_title='no_title'
""").df()
print(exp12.shape)

(87658, 1)


In [73]:
order_reviews['review_comment_message']=order_reviews['review_comment_message'].fillna('no_comment')
order_reviews['review_comment_message']=order_reviews['review_comment_message'].replace(['',' ','  '],'no_comment')
blank = order_reviews['review_comment_message'].str.strip().eq('')
order_reviews.loc[blank, 'review_comment_message'] = 'no_comment'
print((order_reviews['review_comment_message'].str.strip()=='').sum())

0


In [74]:
exp13=db.query("""
select review_comment_message
from order_reviews
where review_comment_message='no_comment'
""").df()
print(exp13.shape)

(58274, 1)


In [75]:
print((order_reviews['review_comment_title'].str.strip() == '').sum())
print((order_reviews['review_comment_message'].str.strip() == '').sum())
print(order_reviews.shape)   

0
0
(99224, 7)


In [76]:
print((order_reviews['review_comment_message'] == 'no_comment').sum())

58274


In [77]:
exp14=db.query("""
select p.product_category_name
from products p 
left join category_name_translations c on c.product_category_name=p.product_category_name
where c.product_category_name_english is null
""").df()
print(exp14.shape)

(623, 1)


In [78]:
exp15=db.query("""
select p.product_category_name,count(p.product_category_name) as count
from products p 
left join category_name_translations c on c.product_category_name=p.product_category_name
where c.product_category_name_english is null
group by p.product_category_name
order by count desc
""").df()
exp15.head()

,product_category_name,count
0,unknown,610
1,portateis_cozinha_e_preparadores_de_alimentos,10
2,pc_gamer,3


In [79]:
exp16=db.query("""
select distinct product_category_name
from products

""").df()
print(exp16.shape)

(74, 1)


In [80]:
exp17=db.query("""
select distinct product_category_name_english
from category_name_translations
""").df()
print(exp17.shape)

(71, 1)


In [81]:
translation_table=pd.DataFrame({'product_category_name':['pc_gamer',
                              'portateis_cozinha_e_preparadores_de_alimentos','unknown'],
                               'product_category_name_english':['pc_gamer',
                                      'portable_kitchen_and_food_preparers','unknown']})
category_name_translations=pd.concat([category_name_translations,translation_table],ignore_index=True)
category_name_translations=category_name_translations.drop_duplicates('product_category_name').reset_index(drop=True)
print(category_name_translations.shape)

(74, 2)


In [82]:
category_name_translations=category_name_translations.drop_duplicates('product_category_name').reset_index(drop=True)
print(category_name_translations.shape)

(74, 2)


In [83]:
print(len(category_name_translations))
print(db.query("select count(*) from category_name_translations").df())

74
   count_star()
0            74


In [84]:
exp18=db.query("""
select p.product_category_name,count(p.product_category_name) as count
from products p 
left join category_name_translations c on c.product_category_name=p.product_category_name
where c.product_category_name_english is null
group by p.product_category_name
order by count desc
""").df()
print(exp18.shape)

(0, 2)


In [85]:
exp19=db.query("""
select product_category_name
from products
group by product_category_name
""").df()
print(exp19.shape)

(74, 1)


In [86]:
exp20=db.query("""
select product_category_name_english
from category_name_translations
group by product_category_name_english
""").df()
print(exp20.shape)

(74, 1)


In [87]:
exp21=db.query("""
select p.product_category_name,c.product_category_name_english
from products p
inner join category_name_translations c on c.product_category_name=p.product_category_name
""").df()
print(exp21.shape)

(32951, 2)


In [88]:
orders = orders.rename(columns={
    'canceled_but_has_date': 'canceled_with_delivery_date',
    'delivered_but_has_nodate': 'delivered_missing_delivery_date',
    'carrier_date_before_delivered_date': 'delivered_before_carrier_pickup'
})

In [89]:
products = products.rename(columns={
    'missed_category_name': 'missing_product_details'
})

In [90]:
orders['delivered_missing_carrier_date'] = (
    np.where((orders['order_delivered_customer_date'].notna()) &
    (orders['order_delivered_carrier_date'].isna()),'yes','no'))

In [91]:
print((orders['canceled_with_delivery_date'] == 'yes').sum())
print((orders['delivered_missing_delivery_date'] == 'yes').sum())
print((orders['delivered_before_carrier_pickup'] == 'yes').sum())
print((products['missing_product_details'] == 'yes').sum())
print((products['missing_dimensions'] == 'yes').sum())

6
8
23
610
2


In [92]:
print((orders['delivered_missing_carrier_date'] == 'yes').sum())  
print(orders.shape)                                                

1
(99441, 13)


In [93]:
print((order_reviews['review_comment_title'].str.strip()=='').sum())
print((order_reviews['review_comment_message'].str.strip()=='').sum())

0
0


In [94]:
tables['orders'] = orders
tables['products'] = products
tables['order_reviews'] = order_reviews
tables['category_name_translations'] = category_name_translations

In [95]:
print(tables['orders'].shape)          
print(tables['products'].shape)       
print(tables['order_reviews'].shape)   
print(tables['category_name_translations'].shape)   

(99441, 13)
(32951, 11)
(99224, 7)
(74, 2)


In [96]:
for name,table in tables.items():
    print(name,table.shape)

customers (99441, 5)
sellers (3095, 4)
order_reviews (99224, 7)
order_items (112650, 7)
products (32951, 11)
geolocations (1000163, 5)
category_name_translations (74, 2)
orders (99441, 13)
order_payments (103886, 5)


In [97]:
for name,table in tables.items():
    table.to_csv(name+'_clean.csv',index=False)
    print(name,table.shape)

customers (99441, 5)
sellers (3095, 4)
order_reviews (99224, 7)
order_items (112650, 7)
products (32951, 11)
geolocations (1000163, 5)
category_name_translations (74, 2)
orders (99441, 13)
order_payments (103886, 5)


In [98]:
import os
os.makedirs('cleaned',exist_ok=True)
for name,table in tables.items():
      table.to_csv('cleaned/'+name+'clean.csv',index=False)
      print(name,table.shape)

customers (99441, 5)
sellers (3095, 4)
order_reviews (99224, 7)
order_items (112650, 7)
products (32951, 11)
geolocations (1000163, 5)
category_name_translations (74, 2)
orders (99441, 13)
order_payments (103886, 5)


In [99]:
import shutil
shutil.make_archive('cleaned_files', 'zip', 'cleaned')

'/kaggle/working/cleaned_files.zip'